# EpistemicTrap-Metacog: Confabulation Retrospection (CR)

**License:** CC0 (Public Domain Dedication)

## 1. Goal & Faculty Isolation
**DeepMind Cognitive Faculty:** Section 7.7.3 (Metacognitive Monitoring - Error Detection).

This benchmark isolates the model's ability to monitor a reasoning process for factual or logical discrepancies and correct them. We present the model with a simulated reasoning transcript containing a single, subtle, verifiably incorrect statement.

### 2. Methodology & Reproducibility (Competition Rule 2.8)
**Data Collection:** Simulated reasoning outputs (N=80) injected with zero-ambiguity factual errors.
**Audit Mechanism:** Deterministic scoring via structured JSON parsing: correct error_location and correct correction.
**Hardware:** Standard Kaggle GPU/CPU kernel.

### Novel Insight
**What can we see now?** We distinguish between "Semantic Fluency" (talking nicely about an error) and "Logical Vigilance" (identifying the error). This reveals whether the model's self-correction system is active or if it is merely a predictive engine biased toward the current context.

---
**Kaggle Domain:** Multi-Document Fact-Checking and Evidence Synthesis  
**Primary Track:** Metacognition (Track 2)  
**Task Name:** EpistemicTrap-Metacognition-CR

### Step 1: Scientific Environment Setup
We initialize the Kaggle Benchmarks SDK for professional scoring.

In [ ]:
# EpistemicTrap CR - Setup
%pip install -q matplotlib seaborn
import os
os.environ['RENDER_SUBRUNS'] = 'False'
import kaggle_benchmarks as kbench
import json
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid")
MODELS_TO_TEST = [kbench.llm]

### Step 2: Zero-Ambiguity Error Dataset
We use items with distinct factual errors to ensure verifiable correctness (AGI Evaluation Criterion).

In [ ]:
import glob
json_paths = glob.glob('/kaggle/input/**/*.json', recursive=True)
dataset_path = next((p for p in json_paths if 'metacog_dataset.json' in p), None)
if not dataset_path: raise FileNotFoundError('Could not find metacog_dataset.json in /kaggle/input/')
with open(dataset_path) as f:
    dataset = json.load(f)
df = pd.DataFrame(dataset)
df_cr = df[df['subtype'] == 'CR'].reset_index(drop=True)

print(f'Loaded Factual Vigilance set: {len(df_cr)} checks.')

### Step 3: Fractional Vigilance Scoring (The Gradient)
Scoring rewards partial success in the "Self-Correction Loop": detection, explanation, and correction.

In [ ]:
@kbench.task(name='EpistemicTrap-Metacognition-CR', version=3)
def metacog_cr(llm, prompt: str, error_location: str, correction: str, **kwargs) -> float:
    try:
        response = llm.prompt(prompt)

        m = re.search(r"\{[\s\S]*\}", response)
        obj = None
        if m:
            try:
                obj = json.loads(m.group(0))
            except Exception:
                obj = None

        pred_loc = None
        pred_corr = None
        if isinstance(obj, dict):
            pred_loc = str(obj.get('error_location', '')).strip()
            pred_corr = str(obj.get('correction', '')).strip()

        loc_ok = 1.0 if pred_loc == str(error_location).strip() else 0.0
        if loc_ok == 0.0 and pred_loc:
            if pred_loc.upper() == str(error_location).strip().upper():
                loc_ok = 1.0

        def norm(s: str) -> str:
            return re.sub(r'\s+', ' ', s.strip().lower())

        # v3: flexible correction matching via accepted_corrections + numeric fallback
        corr_ok = 0.0
        if pred_corr:
            pred_norm = norm(pred_corr)
            # Check exact match
            if pred_norm == norm(str(correction)):
                corr_ok = 1.0
            # Check if gold correction is contained in prediction
            elif norm(str(correction)) in pred_norm:
                corr_ok = 1.0
            else:
                # Check accepted_corrections list
                accepted = kwargs.get('accepted_corrections', [])
                if isinstance(accepted, list):
                    for variant in accepted:
                        if norm(str(variant)) in pred_norm:
                            corr_ok = 1.0
                            break
                # Numeric extraction fallback for partial credit
                if corr_ok == 0.0:
                    gold_nums = set(re.findall(r'\d+\.?\d*', str(correction)))
                    pred_nums = set(re.findall(r'\d+\.?\d*', pred_corr))
                    if gold_nums and gold_nums & pred_nums:
                        corr_ok = 0.5  # partial credit for correct number

        return 0.5 * loc_ok + 0.5 * corr_ok
        
    except Exception as e:
        print(f"Critical Audit Error: {e}")
        return 0.0

### Step 4: Execution at Scale
Measuring factual vigilance across line-labeled single-error transcripts (N=80).

In [ ]:
print(f"Evaluating EpistemicTrap-Metacognition-CR [N={len(df_cr)}]...")
with kbench.client.enable_cache():
    runs = metacog_cr.evaluate(
        llm=MODELS_TO_TEST,
        evaluation_data=df_cr,
    )

### Step 5: Professional Profiling
A Boxplot reveals the model's range of vigilance.

In [ ]:
try:
    df_res = runs.as_dataframe()
    df_res['score'] = df_res['result'].apply(lambda x: float(x) if x is not None else 0.0)
    
    plt.figure(figsize=(12, 6))
    # Visualizing the Vigilance Box Plot
    sns.boxplot(data=df_res, x='score', hue='score', palette='magma', legend=False)
    
    plt.title('Cognitive Profile: Confabulation Retrospection (CR)', fontsize=16, pad=20)
    plt.xlabel('Vigilance Score (Average: {:.2f})'.format(df_res['score'].mean()), fontsize=12)
    plt.xlim(-0.1, 1.1)
    plt.show()
except Exception as e:
    print('Analytics error:', e)

%choose EpistemicTrap-Metacognition-CR